In [1]:
# Colab dependency conflicts require a workaround installing dependencies. Don't ask...
# !pip install "unstructured[pdf]" --no-deps
# !pip install emoji psutil backoff dataclasses-json filetype langdetect python-iso639 python-magic python-oxmsg python-oxmsg rapidfuzz unstructured-client

# !pip install sentence-transformers # Avoid overriding Colab's CUDA libraries. Remove --no-deps when installing locally.
# !pip install sentence-transformers --no-deps

# !pip install pymupdf
# !pip install -U langchain-community langchain-text-splitters
# !python -m spacy download en

# !pip install chromadb

In [2]:
import re
import hashlib
from dateutil.parser import parse
from pprint import pprint
from datetime import datetime

In [3]:
DEVICE = "cuda"   # "cpu"

In [4]:
NAIVE_DATE_RE_COMPILED = re.compile(r"[A-Z][a-z]{2,10}\s\d{1,2},\s2024")  # E.g. "December 27, 2024"

## Load a PDF Document

In [5]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyMuPDFLoader

/home/tt3al/.virtualenvs/agentic-course-12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
loader = PyMuPDFLoader("News Digest 2024 - Question of Cities.pdf")
raw_pdf = loader.load()

In [7]:
raw_pdf[0].metadata

{'producer': 'Skia/PDF m135',
 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
 'creationdate': '2025-04-23T16:10:24+00:00',
 'source': 'News Digest 2024 - Question of Cities.pdf',
 'file_path': 'News Digest 2024 - Question of Cities.pdf',
 'total_pages': 82,
 'format': 'PDF 1.4',
 'title': 'News Digest 2024 - Question of Cities',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2025-04-23T16:10:24+00:00',
 'trapped': '',
 'modDate': "D:20250423161024+00'00'",
 'creationDate': "D:20250423161024+00'00'",
 'page': 0}

In [8]:
pdf_title = raw_pdf[0].metadata["title"]

Things to keep in mind:
1. We don't want any overlap of text between any two articles (obviously). Therefore, we need to treat every article as being it's own document and chunk it individually. It looks like we can use the text on the "Read the full story" link as a separator between articles.
2. An article may split mid-page.
3. It would be nice to have the article publish date as metadata.

In [9]:
raw_whole_pdf_text = "\n".join([page.page_content for page in raw_pdf])
len(raw_whole_pdf_text)

207424

Now, we need to split by article.

In [10]:
raw_article_documents_text = raw_whole_pdf_text.split("Read the full story")
len(raw_article_documents_text)

178

In [11]:
raw_article_documents_text[5]

'\nDecember 13, 2024\nWhy low-intensity Cyclone Fengal caused huge\ndestruction\nCyclone Fengal, which made landfall near Puducherry on November 30, claimed 12 lives,\nmostly in Tamil Nadu, damaged numerous properties and destroyed standing crops over\nlarge areas. Compared to many earlier storms, it was a low-intensity storm but left a trail of\ndestruction, reported The Indian Express. From its genesis to landfall, Fengal moved at a\nslow pace and also remained stationary for nearly 12 hours soon after making landfall close to\nPuducherry. Maintaining its intensity as a cyclone, the storm caused heavy rainfall and rough\nwindy conditions over the region. Usually, after landfall, cyclones weaken as they hit\nobstructions and experience friction from buildings and trees. In the case of Fengal, as the\nstorm stayed stationary, the destruction was far more pronounced.\n'

Good. Next step is to create a list of dictionaries with each dictionary containing a single article, it's publishing date and the title of the document it originated from. We'll try to extract the publishing date from each article, and assign it to it and the following articles, until we hit a new publishing date.

For convenience reasons, we'll stick with `LangChain`'s `Document` type for the article content and metadata. It will help avoid all kinds of conversions when we use other `LangChain` services.

In [12]:
article_documents: list[Document] = []
article_date: str = ""

for raw_article_text in raw_article_documents_text:

    # The articles are ordred by date. The date only shows next to the first article from that date.
    new_article_date: re.Match|None = NAIVE_DATE_RE_COMPILED.search(raw_article_text)
    if new_article_date:
        article_date = new_article_date.group()

    article_documents.append(
        Document(
            page_content=raw_article_text,
            metadata={
                "publish-date": article_date,
                "doc-title": pdf_title,
            }
        )
    )

len(article_documents)

178

In [13]:
print(article_documents[5])

page_content='
December 13, 2024
Why low-intensity Cyclone Fengal caused huge
destruction
Cyclone Fengal, which made landfall near Puducherry on November 30, claimed 12 lives,
mostly in Tamil Nadu, damaged numerous properties and destroyed standing crops over
large areas. Compared to many earlier storms, it was a low-intensity storm but left a trail of
destruction, reported The Indian Express. From its genesis to landfall, Fengal moved at a
slow pace and also remained stationary for nearly 12 hours soon after making landfall close to
Puducherry. Maintaining its intensity as a cyclone, the storm caused heavy rainfall and rough
windy conditions over the region. Usually, after landfall, cyclones weaken as they hit
obstructions and experience friction from buildings and trees. In the case of Fengal, as the
storm stayed stationary, the destruction was far more pronounced.
' metadata={'publish-date': 'December 13, 2024', 'doc-title': 'News Digest 2024 - Question of Cities'}


## Cleanup and Normalize

In [14]:
from unstructured.cleaners.core import clean, clean_bullets, clean_extra_whitespace

In [15]:
def clean_and_normalize_document(article_document: Document, article_publish_date: str) -> Document:
    
    text = article_document.page_content.replace("SEARCH\n", "").replace("MENU\n", "").replace("News Digest\n", "")
    
    if article_publish_date:
        text = text.replace(article_publish_date, "")
    
    text = clean(text)
    text = clean_bullets(text)
    text = clean_extra_whitespace(text)
    text = text.strip()

    article_document.page_content = text
    
    return article_document

In [16]:
for article_document in article_documents:
    article_publish_date = article_document.metadata["publish-date"]
    clean_and_normalize_document(article_document, article_publish_date=article_publish_date)
    if article_publish_date:
        article_document.metadata["publish-date"] = parse(article_publish_date).isoformat()

In [17]:
print(article_documents[5])

page_content='Why low-intensity Cyclone Fengal caused huge destruction Cyclone Fengal, which made landfall near Puducherry on November 30, claimed 12 lives, mostly in Tamil Nadu, damaged numerous properties and destroyed standing crops over large areas. Compared to many earlier storms, it was a low-intensity storm but left a trail of destruction, reported The Indian Express. From its genesis to landfall, Fengal moved at a slow pace and also remained stationary for nearly 12 hours soon after making landfall close to Puducherry. Maintaining its intensity as a cyclone, the storm caused heavy rainfall and rough windy conditions over the region. Usually, after landfall, cyclones weaken as they hit obstructions and experience friction from buildings and trees. In the case of Fengal, as the storm stayed stationary, the destruction was far more pronounced.' metadata={'publish-date': '2024-12-13T00:00:00', 'doc-title': 'News Digest 2024 - Question of Cities'}


## Chunking

Okay. We have our raw articles cleaned-up, normalized and with metadata added. The next thing to do is split each article into chunks. Each chunk will share the same metadata as the article.

#### Sentence-Based Chunking

In [18]:
from langchain_text_splitters import SpacyTextSplitter

In [19]:
SpacyTextSplitter?

Init signature:
SpacyTextSplitter(
    separator: 'str' = '\n\n',
    pipeline: 'str' = 'en_core_web_sm',
    max_length: 'int' = 1000000,
    *,
    strip_whitespace: 'bool' = True,
    **kwargs: 'Any',
) -> 'None'
Docstring:     
Splitting text using Spacy package.

Per default, Spacy's `en_core_web_sm` model is used and
its default max_length is 1000000 (it is the length of maximum character
this model takes which can be increased for large files). For a faster, but
potentially less accurate splitting, you can use `pipeline='sentencizer'`.
Init docstring: Initialize the spacy text splitter.
File:           ~/.virtualenvs/agentic-course-12/lib/python3.12/site-packages/langchain_text_splitters/spacy.py
Type:           ABCMeta
Subclasses:     

In [20]:
splitter = SpacyTextSplitter(chunk_size=800)  

In [21]:
article_chunks: list[Document] = splitter.split_documents(article_documents)

/home/tt3al/.virtualenvs/agentic-course-12/lib/python3.12/site-packages/spacy/pipeline/lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


`SpacyTextSplitter` does not support chunking with sentence overlapping out-of-the-box, then we'll have to introduce our own:

In [22]:
for article_chunk_index in range(1, len(article_chunks)):
    last_sentence_from_previous = article_chunks[article_chunk_index-1].page_content.split("\n\n")[-1]
    article_chunks[article_chunk_index].page_content = last_sentence_from_previous + "\n\n" + article_chunks[article_chunk_index].page_content

In [23]:
len(article_chunks)

370

In [24]:
print(article_chunks[1])
print("~" * 100)
print(article_chunks[2])

page_content='However, the forest cover grew by just 156 square kilometres and most of the gain (149 square kilometres) occurred outside the Recorded Forest Area (RFA), which refers to areas designated as forests in government records, reports the Business Standard.

Experts claimed that the government counted bamboo plantations, coconut groves, and orchards among others as part of the forest cover and produced “another faulty report with inflated data” and argued that such areas do not offer any ecological value for biodiversity and wildlife conservation.

They said, according to government data, 1,488 square kilometres of unclassed forests were lost between 2021 and 2023 but “there is no explanation for it” in the ISFR 2023.' metadata={'publish-date': '2024-12-27T00:00:00', 'doc-title': 'News Digest 2024 - Question of Cities'}
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
page_content='They said, according to government data, 1,4

Seems like we have our overlapping.

## Vectorizing

In [25]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

In [26]:
CHROMADB_CLIENT = chromadb.Client()

In [27]:
# Embeddings take around 75 seconds on CPU.
bge_ef = SentenceTransformerEmbeddingFunction(model_name="BAAI/bge-small-en-v1.5", device=DEVICE)  # device="cuda"

CHROMADB_COLLECTION = CHROMADB_CLIENT.get_or_create_collection(
    name="augmentation-articles-collection-1",
    embedding_function=bge_ef,   # chromadb.utils.embedding_functions.ONNXMiniLM_L6_V2()
    configuration={
        "hnsw": {
            "space": "cosine"
        },
    }
)

In [28]:
print(f">>> About to ingest {len(article_chunks)} chunks...")

>>> About to ingest 370 chunks...


In [29]:
for chunk_document in article_chunks:
    
    content = chunk_document.page_content
    CHROMADB_COLLECTION.add(
            ids=[hashlib.md5(content.encode()).hexdigest()[-8:]],
            documents=[content],  # Text for embedding
            metadatas=[{
                **chunk_document.metadata,
                "token-count": int(len(content.split()) * 1.4),
            }],
    )

In [30]:
CHROMADB_COLLECTION.count()

367

In [31]:
print(f"Wait a second. Didn't we have {len(article_chunks)} chunks?!")

Wait a second. Didn't we have 370 chunks?!


In [32]:
texts = [chunk.page_content for chunk in article_chunks]

In [33]:
len(set(texts))

367

In [34]:
for i, text in enumerate(texts):
    if text in texts[i+1:]:
        print(f">>> chunk #{i} same as #{texts.index(text, i+1)}") 

>>> chunk #33 same as #46
>>> chunk #34 same as #47
>>> chunk #36 same as #49


In [35]:
article_chunks[33].page_content

'With global temperatures rapidly nearing the 1.5C threshold, we must act now, collectively, to mitigate impending disasters that will affect both mountain populations and 1.6 billion people living downstream.”\n\n‘How did human-rights thrashing Azerbaijan get to host COP 29?’\n\nSwedish activist and international climate crisis campaigner Greta Thunberg wrote, in an opinion column, that the COP 29 summit, like those before, is a mere act of greenwashing.\n\nThe host country Azerbaijan’s entire economy is built on fossil fuels with the state-owned oil company Socar’s oil and gas exports accounting for close to 90 percent of the country’s exports.\n\n“Despite what it might claim, Azerbaijan has no ambition to take climate action.\n\nIt is planning to expand fossil fuel production, which is completely incompatible with the 1.5C limit and the goals of the Paris agreement on climate change.\n\nHow did this country get to host the climate summit,” she questions.'

In [36]:
article_chunks[46].page_content

'With global temperatures rapidly nearing the 1.5C threshold, we must act now, collectively, to mitigate impending disasters that will affect both mountain populations and 1.6 billion people living downstream.”\n\n‘How did human-rights thrashing Azerbaijan get to host COP 29?’\n\nSwedish activist and international climate crisis campaigner Greta Thunberg wrote, in an opinion column, that the COP 29 summit, like those before, is a mere act of greenwashing.\n\nThe host country Azerbaijan’s entire economy is built on fossil fuels with the state-owned oil company Socar’s oil and gas exports accounting for close to 90 percent of the country’s exports.\n\n“Despite what it might claim, Azerbaijan has no ambition to take climate action.\n\nIt is planning to expand fossil fuel production, which is completely incompatible with the 1.5C limit and the goals of the Paris agreement on climate change.\n\nHow did this country get to host the climate summit,” she questions.'

Yep...

## Retrieval

In [37]:
user_query = "Do data centres have the potential to affect climate? Provide examples and numbers."

In [38]:
search_results = CHROMADB_COLLECTION.query(
    query_texts=[user_query],
    n_results=7,
    include=["documents", "metadatas", "distances"],
)

In [39]:
pprint(list(zip(search_results["distances"][0], search_results["documents"][0])))

[(0.22203350067138672,
  'The International Energy Agency stated that data centres already accounted '
  'for 1 percent to 1.5 percent of global electricity consumption in 2022 – '
  'and that was before the AI boom began with ChatGPT’s launch at the end of '
  'that year.\n'
  '\n'
  'all five tech companies have claimed carbon neutrality, though Google '
  'dropped the label last year as it stepped up its carbon accounting '
  'standards.'),
 (0.23691320419311523,
  'Regenvanu said there was an urgent need for a response to climate change '
  'that was rooted in international law rather than politics: “We look to the '
  'court for recognition that the conduct which has already caused immense '
  'harm to my people and so many others is unlawful, that it must cease, and '
  'that its consequences must be repaired.”\n'
  '\n'
  'Data centre in Georgia town poses a climate problem Property owners around '
  'Fayetteville, Georgia, are at the front lines of a battle over '
  'infrastruc

In [40]:
vector_store_chunk_ids = search_results["ids"][0]
vector_store_chunk_ids

['f2758a4f',
 'ced4dc26',
 '8c10ad05',
 '4f76cb0b',
 '4904ccfa',
 'd728e09f',
 'b59b14dd']

### Re-ranking

In [41]:
from sentence_transformers import CrossEncoder

In [42]:
rerank_model = CrossEncoder("mixedbread-ai/mxbai-rerank-large-v1")

In [43]:
reranked_chunks = rerank_model.rank(user_query, search_results["documents"][0], return_documents=True, top_k=7)

In [44]:
reranked_chunks

[{'corpus_id': 3,
  'score': np.float32(0.6695042),
  'text': 'Amazon is the largest emitter of the big five tech companies by a mile – the emissions of the second-largest emitter, Apple, were less than half of Amazon’s in 2022.\n\nAmazon is the largest emitter of the big five tech companies by a mile – the emissions of the second-largest emitter, Apple, were less than half of Amazon’s in 2022.\n\nHowever, Amazon has been kept out of the calculation above because its differing business model makes it difficult to isolate data centre-specific emissions figures for the company.\n\nAs energy demands for these data centres grow, many are worried that carbon emissions will, too.\n\nThe International Energy Agency stated that data centres already accounted for 1 percent to 1.5 percent of global electricity consumption in 2022 – and that was before the AI boom began with ChatGPT’s launch at the end of that year.'},
 {'corpus_id': 0,
  'score': np.float32(0.56746805),
  'text': 'The Internatio

For this exercise, we'll have the threshold score set to `0.3`.

In [45]:
THRESHOLD_SCORE = 0.4

In [46]:
final_augmentation_chunks = [reranked_chunk["text"] for reranked_chunk in reranked_chunks if reranked_chunk["score"] >= THRESHOLD_SCORE]
len(final_augmentation_chunks)

2

In [47]:
final_augmentation_chunk_ids = [
    vector_store_chunk_ids[reranked_chunk["corpus_id"]] for reranked_chunk in reranked_chunks if reranked_chunk["score"] >= THRESHOLD_SCORE
]
final_augmentation_chunk_ids

['4f76cb0b', 'f2758a4f']

## Prompting

In [48]:
STUFFING_PROMPT_TEMPLATE = """
You are an expert assistant. Answer the user's question based MOSTLY on the provided context.

If the context does not contain the answer, say "I don't know based on the context."

Context:
{context}

Question: {user_query}

Answer:
"""

In [49]:
context = "\n---\n".join(final_augmentation_chunks)
print(context)

Amazon is the largest emitter of the big five tech companies by a mile – the emissions of the second-largest emitter, Apple, were less than half of Amazon’s in 2022.

Amazon is the largest emitter of the big five tech companies by a mile – the emissions of the second-largest emitter, Apple, were less than half of Amazon’s in 2022.

However, Amazon has been kept out of the calculation above because its differing business model makes it difficult to isolate data centre-specific emissions figures for the company.

As energy demands for these data centres grow, many are worried that carbon emissions will, too.

The International Energy Agency stated that data centres already accounted for 1 percent to 1.5 percent of global electricity consumption in 2022 – and that was before the AI boom began with ChatGPT’s launch at the end of that year.
---
The International Energy Agency stated that data centres already accounted for 1 percent to 1.5 percent of global electricity consumption in 2022 – 

In [50]:
final_prompt = STUFFING_PROMPT_TEMPLATE.format(context=context, user_query=user_query)
print(final_prompt)


You are an expert assistant. Answer the user's question based MOSTLY on the provided context.

If the context does not contain the answer, say "I don't know based on the context."

Context:
Amazon is the largest emitter of the big five tech companies by a mile – the emissions of the second-largest emitter, Apple, were less than half of Amazon’s in 2022.

Amazon is the largest emitter of the big five tech companies by a mile – the emissions of the second-largest emitter, Apple, were less than half of Amazon’s in 2022.

However, Amazon has been kept out of the calculation above because its differing business model makes it difficult to isolate data centre-specific emissions figures for the company.

As energy demands for these data centres grow, many are worried that carbon emissions will, too.

The International Energy Agency stated that data centres already accounted for 1 percent to 1.5 percent of global electricity consumption in 2022 – and that was before the AI boom began with Cha

## Generation

In [51]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [52]:
HF_LLM_MODEL_ID = "unsloth/Llama-3.2-1B-Instruct"

In [ ]:
LLM_TOKENIZER = AutoTokenizer.from_pretrained(HF_LLM_MODEL_ID)
LLM_MODEL = AutoModelForCausalLM.from_pretrained(HF_LLM_MODEL_ID, dtype=torch.bfloat16, device_map="auto")

In [57]:
GENERATOR = pipeline(task="text-generation", model=LLM_MODEL, tokenizer=LLM_TOKENIZER)

Device set to use cuda:0


In [58]:
response = GENERATOR(final_prompt, max_new_tokens=500, return_full_text=False, temperature=0.01)
print(response[0]["generated_text"])

Yes, data centres have the potential to affect climate. Here are some examples and numbers:

* Data centres are a significant source of greenhouse gas emissions, accounting for around 1-2% of global electricity consumption, which is equivalent to around 1.5-3 billion metric tons of CO2 per year.
* The energy demands for data centres are growing rapidly, with some estimates suggesting that they could account for up to 10% of global electricity consumption by 2030.
* The International Energy Agency (IEA) estimates that data centres will account for 1-2% of global electricity consumption by 2030, which is equivalent to around 1.5-3 billion metric tons of CO2 per year.
* The carbon footprint of data centres can vary widely depending on the type of technology used, the efficiency of the cooling system, and the location of the data centre.
* Some data centres are using renewable energy sources, such as solar or wind power, to reduce their carbon footprint.
* However, many data centres are st

<hr/>